In [131]:
import pandas as pd
import numpy as np

import torch
from torch import nn
from torch import optim
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_squared_error, r2_score

In [132]:
FILE_PATH_TRAIN = "../data/split/train/train.csv"
df_train = pd.read_csv(FILE_PATH_TRAIN)
FILE_PATH_TEST = "../data/split/test/test.csv"
df_test = pd.read_csv(FILE_PATH_TEST)

Hồ Chí Minh chia ra thành 3 khu vực (theo Quyết định 79/2024/QĐ-UBND bảng giá đất TP.HCM):
- Khu vực 1: Quận 1, Quận 3, Quận 4, Quận 5, Quận 6, Quận 10, Quận 11, quận Bình Thạnh, quận Phú Nhuận.
- Khu vực 2: Quận 7, Quận 8, Quận 12, quận Tân Bình, quận Tân Phú, quận Bình Tân, quận Gò Vấp, thành phố Thủ Đức.
- Khu vực 3: Bình Chánh, huyện Hóc Môn, huyện Củ Chi (Không có dữ liệu), huyện Nhà Bè, huyện Cần Giờ (không có dữ liệu).

In [133]:
khu_vuc_1 = [
    'quận 1', 'quận 3', 'quận 4', 'quận 5', 'quận 6',
    'quận 10', 'quận 11', 'bình thạnh', 'phú nhuận'
]

khu_vuc_2 = [
    'quận 7', 'quận 8', 'quận 12', 'tân bình', 'tân phú',
    'bình tân', 'gò vấp', 'thủ đức', 'quận 9', 'quận 2'
]

khu_vuc_3 = [
    'bình chánh', 'nhà bè', 'cần giờ'
]

In [134]:
region_mapping = {}
region_mapping.update({quan: 'Khu vực 1' for quan in khu_vuc_1})
region_mapping.update({quan: 'Khu vực 2' for quan in khu_vuc_2})
region_mapping.update({quan: 'Khu vực 3' for quan in khu_vuc_3})

In [135]:
df_train['region'] = df_train['address'].map(region_mapping)
df_test['region'] = df_test['address'].map(region_mapping)

In [136]:
def assign_region(addr):
    if addr in khu_vuc_1:
        return 'Khu vực 1'
    if addr in khu_vuc_2:
        return 'Khu vực 2'
    if addr in khu_vuc_3:
        return 'Khu vực 3'
    return 'Khác'

df_train['region'] = df_train['address'].apply(assign_region)
df_test['region'] = df_test['address'].apply(assign_region)


In [137]:
region_map = {
    'Khu vực 1': 1,
    'Khu vực 2': 2,
    'Khu vực 3': 3,
    'Khác': 0
}

df_train['region_encoded'] = df_train['region'].map(region_map)
df_test['region_encoded'] = df_test['region'].map(region_map)

In [138]:
X_train = df_train[['area', 'bedrooms', 'bathrooms', 'region_encoded']]
y_train = df_train['price']
X_test = df_test[['area', 'bedrooms', 'bathrooms', 'region_encoded']]
y_test = df_test['price']

In [139]:
scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_test_s = scaler_X.transform(X_test)

y_train_s = y_train.values.reshape(-1, 1)
y_test_s = y_test.values.reshape(-1, 1)


In [140]:
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_test_t = torch.tensor(y_test_s, dtype=torch.float32)

In [141]:
model = nn.Linear(X_train_t.shape[1], 1)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) 

In [142]:
epochs = 10000
train_losses = []

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    y_pred_train = model(X_train_t)
    loss = criterion(y_pred_train, y_train_t)
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    
    if (epoch+1) % 1000 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {loss.item():.6f}")


Epoch [1000/10000] - Loss: 128.215836
Epoch [2000/10000] - Loss: 106.340477
Epoch [3000/10000] - Loss: 91.761719
Epoch [4000/10000] - Loss: 80.529488
Epoch [5000/10000] - Loss: 71.710434
Epoch [6000/10000] - Loss: 65.177223
Epoch [7000/10000] - Loss: 60.685841
Epoch [8000/10000] - Loss: 57.879169
Epoch [9000/10000] - Loss: 56.342564
Epoch [10000/10000] - Loss: 55.669315


In [143]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test_t).squeeze().cpu().numpy()

y_test_orig = y_test.values


In [144]:
r2 = r2_score(y_test_orig, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred))
print(f"R2: {r2:.4f}, RMSE: {rmse:.4f}")

R2: 0.5430, RMSE: 7.7252


In [146]:
print("\n=== MODEL PARAMETERS ===")
print("Weights:", model.weight.detach().numpy())
print("Bias:", model.bias.detach().numpy())


=== MODEL PARAMETERS ===
Weights: [[ 5.1160994  -0.3521494   1.5527151  -0.44135034]]
Bias: [7.745484]
